# Round 2: fine-tune the colony detector on cleaned labels (Kaggle)

Round 1 (`train_rfdetr_kaggle.ipynb`) learnt from the original labels, which miss about 6% of real
colonies and box some stickers, scratches and pin-point specks. This round:

1. drops labels below 0.3 mm (counting rule B ignores them),
2. adds colonies the round-1 model is very sure of (confidence >= 0.9) that the labels missed,
3. fine-tunes the round-1 model on that for 6 epochs,
4. scores it on the **reviewed answer key** (`eval/gold_test_ruleB`, 138 test plates) and fits the cut-offs.

**Before running:**
- Settings: Accelerator **GPU T4 x2** (or P100), Internet **On**.
- Give it the round-1 model, either way:
  - **Add Input → Your Work → Notebooks →** the round-1 notebook (its output has `rfdetr_tiles.pth`), or
  - upload `rfdetr_tiles.pth` to a GitHub release tagged `model-v1` in this repo.

Then **Save Version → Save & Run All (Commit)**. About 6–7 hours. Results land in **Output**:
`rfdetr_tiles.pth`, `detector.json`, `gold_report.txt`, `gold_results.csv`, logs.

In [ ]:
# 1. GPU, code, packages
import torch, os
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
OUT = '/kaggle/working'
!rm -rf /kaggle/temp/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /kaggle/temp/colony_detector
%cd /kaggle/temp/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml

In [ ]:
# 2. Dataset (~3.2 GB) from the GitHub release
!mkdir -p /kaggle/temp/data && cd /kaggle/temp/data && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.zip && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.2.zip && \
  unzip -q -n data.set.zip -d raw && unzip -q -n data.set.2.zip -d raw && rm -f *.zip && ls raw

In [ ]:
# 3. The round-1 model
import glob, subprocess
found = glob.glob('/kaggle/input/**/rfdetr_tiles.pth', recursive=True)
PREV = '/kaggle/temp/prev_rfdetr_tiles.pth'
if found:
    subprocess.run(['cp', found[0], PREV], check=True)
else:
    subprocess.run(['wget', '-q', '-O', PREV,
                    'https://github.com/akashbabu9392/colony_detector/releases/download/model-v1/rfdetr_tiles.pth'])
assert os.path.getsize(PREV) > 50_000_000, 'Round-1 model not found: add it as Input or upload it to release model-v1'
print('round-1 model:', found[0] if found else 'release model-v1', os.path.getsize(PREV) // 2**20, 'MB')

In [ ]:
# 4. Tiles with cleaned labels: drop < 0.3 mm, add confident missed colonies (~1 h: runs the model on every plate)
!python tools/build_training_set.py --dataset /kaggle/temp/data/raw --out /kaggle/temp/tiles --scale 1.0 \
    --empty-keep 0.15 --coco --gold "" --min-label-mm 0.3 --repair-weights {PREV} --repair-conf 0.9 \
    2>&1 | grep -E "label cleaning|tiles:|Error|Traceback" | tee {OUT}/clean_log.txt

In [ ]:
# 5. Fine-tune from the round-1 model: 6 epochs x ~45 min on a T4
!python tools/train_rfdetr.py --dataset /kaggle/temp/tiles/coco --size medium --resolution 640 --init {PREV} \
    --epochs 6 --batch 4 --grad-accum 4 --lr 5e-5 --output /kaggle/temp/rfdetr --out models/rfdetr_tiles.pth \
    2>&1 | grep -E --line-buffered "Epoch|Best|mAP|Error|Traceback|->" | tee {OUT}/train_log.txt
!cp models/rfdetr_tiles.pth {OUT}/

In [ ]:
# 6. Score on the reviewed answer key (rule B) and fit the cut-offs (cross-validated estimate printed too)
os.environ['CD_RFDETR_WEIGHTS'] = 'models/rfdetr_tiles.pth'
os.environ['CD_ENGINES'] = 'rfdetr'
!python tools/score_gold.py --gold eval/gold_test_ruleB --images /kaggle/temp/data/raw \
    --cache /kaggle/temp/test_dets.pkl --tune models/detector.json 2>/dev/null | tee {OUT}/gold_report.txt
!python tools/score_gold.py --gold eval/gold_test_ruleB --cache /kaggle/temp/test_dets.pkl \
    --detector models/detector.json --csv {OUT}/gold_results.csv > /dev/null
!cp models/detector.json /kaggle/temp/test_dets.pkl {OUT}/ && ls -la {OUT}